# 26 — PDF Parsing
**Goal:** Extract text from PDF resumes while preserving layout.

PDF is the format resumes most often arrive in — and the hardest to parse: the file describes *where glyphs are drawn on a page*, not what the text says. This chapter builds the PDF branch of the extraction pipeline: create a small resume PDF, pull its text back out with `pdfplumber`, and think about what multi-column layouts do to reading order.

**Why it matters for resumes / ATS:** an ATS must extract text before it can index skills or rank candidates. If extraction scrambles column order or silently drops text, every downstream stage — normalization (Ch. 29), language routing (Ch. 30), section detection (Ch. 32) — inherits the damage. Getting the text out in reading order here decides how much of the resume the rest of the pipeline ever sees.

![Document Parsing Pipeline](../../../assets/images/document_parsing_pipeline_1785491201751.png)

> **Figure:** The Document Parsing Pipeline supporting PDF, DOCX and scanned images as input formats — all feeding the same normalization layer.

Three input formats, one exit: every branch of this pipeline must produce the same thing — clean, linear text for the normalizer. Chapters 26–28 cover the three branches in turn (PDF here, DOCX next, scanned images via OCR after that); the normalization layer they all feed into is Ch. 29.

## 1. Why PDF is Hard

A PDF is a page-description format, not a text format: the file stores positioned glyphs and drawing commands, with no concept of paragraphs, reading order, or even a guaranteed text layer. Extraction therefore becomes a geometry problem — reconstructing sentences from coordinates.

**What the code does:** prints the standard hazard list for resume parsing:
- No inherent text flow — text is placed line by line, so paragraphs must be re-assembled
- Multi-column layouts — reading order must be recovered from x/y positions
- Tables, headers, footers — content that must be detected and handled deliberately
- Embedded fonts — custom encodings can map glyphs to the wrong Unicode codepoints
- Scanned PDFs — no text layer at all, so the file falls through to OCR (Ch. 28)

It also names the tool tiers: `pdfplumber` (layout-aware, the workhorse for resumes), PyMuPDF (`fitz`, very fast), and `pdfminer` (low-level building blocks). Expected output: the five challenges, then the tools line.

In [ ]:
print('''PDF challenges for resume parsing:\n1. No inherent text flow\n2. Multi-column layouts\n3. Tables, headers, footers\n4. Embedded fonts\n5. Scanned PDFs need OCR\n\nTools: pdfplumber (layout-aware), PyMuPDF (fast), pdfminer (low-level)''')

## 2. Extracting Text with pdfplumber

This cell does a full round-trip: it *generates* a minimal resume PDF with `fpdf`, then reopens it with `pdfplumber` and calls `page.extract_text()` — the layout-aware method that merges glyphs into lines using their positions on the page.

**What the code does:**
- Builds the PDF: a centered name, the email, then bold `PROFESSIONAL SUMMARY` / `EXPERIENCE` headers with body text (`multi_cell` wraps the summary line)
- Saves to `/tmp/test_resume.pdf` and prints the file size in bytes
- Reopens with `pdfplumber.open()` and prints the first 200 characters of each page's extracted text

**Expected:** with the em dash handled, the size prints around 1.3 KB and `Page 1:` is followed by the text in reading order — `Srivatsa Gorti`, the email, the section headers, and the summary line. One version-sensitive trap: `fpdf2 >= 2.7.8` substitutes `Arial` with the built-in Latin-1 `Helvetica`, so the em dash in "Google — Senior Data Scientist" raises `FPDFUnicodeEncodingException` when saving. Replace the dash (or register a Unicode TTF) and the cell runs as described.

In [ ]:
from fpdf import FPDF
import pdfplumber, os

pdf = FPDF()
pdf.add_page()
pdf.set_font("Arial", size=12)
pdf.cell(200, 10, text="Srivatsa Gorti", new_x="LMARGIN", new_y="NEXT", align="C")
pdf.set_font("Arial", size=10)
pdf.cell(200, 10, text="srivatsa@email.com", new_x="LMARGIN", new_y="NEXT", align="C")
pdf.set_font("Arial", style="B", size=11)
pdf.cell(200, 10, text="PROFESSIONAL SUMMARY", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("Arial", size=10)
pdf.multi_cell(0, 5, text="Data scientist with 5+ years of Python, NLP, and ML experience.")
pdf.set_font("Arial", style="B", size=11)
pdf.cell(200, 10, text="EXPERIENCE", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("Arial", size=10)
pdf.cell(200, 10, text="Google — Senior Data Scientist, 2020-Present", new_x="LMARGIN", new_y="NEXT")

test_pdf = "/tmp/test_resume.pdf"
pdf.output(test_pdf)
print(f"PDF created: {os.path.getsize(test_pdf)} bytes")

with pdfplumber.open(test_pdf) as pdfp:
    for i, page in enumerate(pdfp.pages):
        text = page.extract_text() or ""
        print(f"\nPage {i+1}:\n{text[:200]}")

## 3. Multi-Column Layout Handling

A two-column resume is the classic pdfplumber failure: naive extraction walks glyphs in file order, scrambling the two columns together. The fix is to treat extraction as a **layout problem** — cluster words by horizontal position, then read columns top-to-bottom, left-to-right.

**What the code does:** prints the standard column-recovery strategy:
1. Extract words with their bounding boxes (`extract_words()` gives `x0`, `top`, ...)
2. Cluster by x-coordinate into columns
3. Sort by y within each column
4. Merge columns left → right into reading order

**Try it:** `extract_text()` already handles *simple* multi-column pages; for complex ones (nested tables, sidebars) you drop to per-character extraction and implement the clustering yourself. Expected output: the four numbered steps, then the two tooling notes about `extract_text()` and per-character extraction.

In [ ]:
print('''Multi-column strategy:\n1. Extract words with bounding boxes\n2. Cluster by x-coordinate -> columns\n3. Sort by y within columns\n4. Merge left->right reading order\n\npdfplumber.extract_text() handles basic multi-column.\nFor complex layouts, use pdfplumber's per-character extraction.''')

## Summary: Use pdfplumber for layout-aware extraction. PyMuPDF for speed.

**PDF parsing is a layout-reconstruction problem, not a text-reading one.** Choose `pdfplumber` when reading order matters (it always does for resumes) and PyMuPDF when raw speed beats fidelity. Whatever the tool, expect noisy output — stray headers, footer page numbers, column interleaving — which is exactly the mess the normalizer in Ch. 29 is built to clean.

Next up: the easy case. Where PDFs hide their structure, `.docx` files (Ch. 27) carry it explicitly — paragraphs, styles, and tables come back as first-class objects instead of positioned glyphs.